In [18]:
import { createAgent } from "npm:langchain";
import { ChatGoogle } from "npm:@langchain/google";
import { parse } from "jsr:@std/dotenv";
import { tool } from "npm:langchain";
import { z } from "npm:zod";


In [19]:
// env파일 불러오기
const env = parse(await Deno.readTextFile(".env"));

const GOOGLE_API_KEY = env.GOOGLE_API_KEY;

const API_URL = "https://www.daegufood.go.kr/kor/api/tasty.html?mode=json";

In [20]:
const model = new ChatGoogle({
  model: "gemini-3.1-flash-lite",
  apiKey: GOOGLE_API_KEY,
  temperature: 0,
});


restaurantSearchTool 함수 역할

사용자가 실제 식당을 찾는다
       ↓
대구 지역 확인
       ↓
공공데이터 API 호출
       ↓
카테고리 필터
       ↓
식당 목록 반환

In [21]:
type RestaurantCategory =
  | "한식"
  | "일식"
  | "중식"
  | "양식"
  | "세계요리"
  | "특별한 술집"
  | "전통차/커피전문점"
  | "디저트/베이커리";

type Restaurant = {
  OPENDATA_ID: string;
  BZ_NM: string;
  GNG_CS: string;
  FD_CS: string;
  TLNO: string;
  MBZ_HR: string;
  SEAT_CNT: string;
  PKPL: string;
  HP: string;
  PSB_FRN: string;
  BKN_YN: string;
  INFN_FCL: string;
  BRFT_YN: string;
  DSSRT_YN: string;
  MNU: string;
  SMPL_DESC: string;
  SBW: string;
  BUS: string;
};


// 외부api 호출 함수
async function fetchRestaurants(district: string): Promise<Restaurant[]> {
  const url = new URL(API_URL);

  url.searchParams.set("addr", district);

  const response = await fetch(url);

  if (!response.ok) {
    throw new Error(`API 요청 실패: ${response.status}`);
  }

  const data = await response.json();

  // 실제 API 응답 구조에 맞게 수정
  return data.data as Restaurant[];
}

// 카테고리 필터 함수
function filterByCategory(
  restaurants: Restaurant[],
  category: RestaurantCategory | null,
): Restaurant[] {
  if (category === null) {
    return restaurants;
  }

  return restaurants.filter((restaurant) => restaurant.FD_CS === category);
}

// 지역,카테고리에 맞는 음식점 검색 함수
async function searchRestaurants(
  district: string,
  category: RestaurantCategory | null,
) {
  const restaurants = await fetchRestaurants(district);

  const filtered = filterByCategory(restaurants, category);

  return filtered;
}


const restaurantSearchTool = tool(
  // 실제로 실행할 함수
  async ({ district, category }) => {
    const restaurants = await searchRestaurants(district, category);

    return {
      count: restaurants.length,

      restaurants: restaurants.map((restaurant) => ({
        name: restaurant.BZ_NM,

        category: restaurant.FD_CS,

        address: restaurant.GNG_CS,

        menu: restaurant.MNU,

        parking: restaurant.PKPL,

        reservation: restaurant.BKN_YN,

        seats: restaurant.SEAT_CNT,

        description: restaurant.SMPL_DESC,
      })),
    };
  },
  // tool의 설명
  {
    name: "search_restaurants",

    description:
      "대구광역시의 특정 지역에서 음식점, 카페, 술집 등을 공공데이터 API로 검색합니다.",

    schema: z.object({
      district: z
        .string()
        .describe("검색할 대구광역시 행정구역. 예: 남구, 중구, 수성구"),

      category: z
        .enum([
          "한식",
          "일식",
          "중식",
          "양식",
          "세계요리",
          "특별한 술집",
          "전통차/커피전문점",
          "디저트/베이커리",
        ])
        .nullable()
        .describe("검색할 식당 카테고리. 특정 카테고리가 없으면 null"),
    }),
  },
);



에이전트 만들기
4단계과정을 에이전트로 생성하여 쉽게 호출 가능
Agent를 LLM과 Tool을 결합해 작업을 추론하고, Tool을 선택하며, 목표를 향해 반복적으로 실행하는 시스템

In [22]:
const agent = createAgent({
  model,

  tools: [restaurantSearchTool],

  systemPrompt: `
당신은 대구 맛집을 추천하는 AI 에이전트입니다.

사용자가 실제 음식점, 카페, 술집 등을 찾거나
추천을 요청하면 search_restaurants Tool을 사용하여
공공데이터를 확인하세요.

실제 식당 정보가 필요한 질문에는
모델의 기억만으로 식당 정보를 만들어내지 마세요.

사용자가 음식에 대한 일반적인 설명이나
개념을 질문하는 경우에는 Tool을 사용할 필요가 없습니다.

Tool에서 반환된 정보만을 근거로
간결하게 답변하세요.
`,
});


In [23]:
const result = await agent.invoke({
  messages: [
    {
      role: "user",
      content: "대구 남구에서 디저트 먹을만한곳 추천해줘",
    },
  ],
});

console.log(result.messages.at(-1).content)


대구 남구에서 방문하기 좋은 디저트 맛집 3곳을 추천해 드립니다.

1. **구민회의 좋은아침 (대명동)**
   - 정직한 재료로 매일 아침 빵을 굽는 베이커리입니다. 샌드위치, 식빵, 타르트, 케이크 등 다양한 메뉴가 준비되어 있습니다.

2. **더셰프 (이천동)**
   - 가족이 먹는다는 마음으로 정직하게 만드는 베이커리입니다. 쉐프버거, 밤식빵, 도넛, 앙버터파이 등을 맛볼 수 있습니다.

3. **프라리네 (대명동)**
   - 2007년부터 운영 중인 파이 및 케이크 전문점입니다. 치즈, 호두, 블루베리, 무화과 파이와 당근케이크 등이 인기 메뉴입니다.
